## The Need for Sequence Models (Text and Speech)

A plain fully connected network, referred to here as a DNN, needs an input of one fixed
size. Text and speech do not come in one fixed size:

- A sentence can have 3 words or 300 words.
- A spoken word can last 0.3 s or 1 s, and it can start anywhere inside a recording.

The usual workaround is to force every example to the same length:

- **Padding**: append (or prepend) a dummy `<pad>` symbol or silent frames until the
  sequence reaches `MAX_LEN`.
- **Truncating**: cut away everything after (or before) `MAX_LEN`.
- **Averaging** (bag of words / bag of frames): average all the vectors into one
  fixed-size vector, which removes length altogether.

This notebook shows, with experiments, why each of these workarounds breaks down, and
then builds a **pure sequence model**: a recurrent network that reads one token (or one
audio frame) at a time with the *same* weights at every step, and that never sees a
single padding symbol.

We study four limitations of the DNN approach, all measured below.

1. **Truncation throws information away.** If the important word is past `MAX_LEN`,
   the DNN never sees it.
2. **Every position has its own weights.** A DNN has to learn the same rule separately
   at position 3, position 17, and position 42. Positions that were always padding
   during training are never learned.
3. **Averaging destroys order.** "not good ... cheap" and "good ... not cheap" contain the
   same words. "yes" and "say" contain the same sounds.
4. **Parameters grow with `MAX_LEN`.**

To keep the notebook self contained and fast on a laptop CPU, both datasets are small
synthetic datasets that are built to mirror a real property of language and speech.
The rules that generate them are fully visible, so every result can be traced back to
a cause.

In [ ]:
import time
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

In [ ]:
torch.manual_seed(0)
np.random.seed(0)
rng = np.random.default_rng(0)

device = torch.device("cpu")
print("device:", device)

### 1. Text dataset: short reviews with negation

Each example is a short, restaurant-review-like sentence with **exactly one** opinion
word (such as `tasty` or `rude`). The opinion word can appear at **any** position. Half
of the time it is negated by the word `not` directly in front of it.

The label is the meaning of the sentence:

| sentence fragment | label |
| --- | --- |
| `... was tasty ...` | positive |
| `... was not tasty ...` | negative |
| `... was rude ...` | negative |
| `... was not rude ...` | positive |

To make word order matter, some sentences also contain a **distractor** `not` in front of
an ordinary word (`not cheap`, `not busy`). A distractor does not change the label. So the
model must notice *where* the `not` is, not just *whether* it appears.

Sentence lengths vary, as they do in real text.

In [ ]:
FILLER = (
    "the food service was and we it place staff really quite very also today our "
    "visit menu price room is a this that were for with at to of my table evening "
    "lunch waiter drinks music view cheap busy"
).split()
POSITIVE = "great good excellent amazing lovely tasty friendly delicious".split()
NEGATIVE = "bad terrible awful rude bland dirty slow disappointing".split()

PAD = "<pad>"

In [ ]:
vocab = [PAD, "not"] + FILLER + POSITIVE + NEGATIVE
word2id = {w: i for i, w in enumerate(vocab)}

# word2id

In [ ]:
PAD_ID, NOT_ID = word2id[PAD], word2id["not"]
V = len(vocab)
print(f"vocabulary size: {V} (id 0 is {PAD!r})")

**Text generation technique(rule-based synthetic data):**

1. Start with a random sentence length and fill all positions with neutral words
    from FILLER ({len(FILLER)} tokens).

2. Choose sentiment polarity uniformly:
    - positive opinion word from POSITIVE ({len(POSITIVE)} words), or
    - negative opinion word from NEGATIVE ({len(NEGATIVE)} words).

3. Place exactly one opinion word at a random position (not the first token).

4. With 50% probability, negate that opinion by putting 'not' directly before it.
    This flips the final meaning label.

5. Add 0–2 distractor 'not' tokens at other positions (never right before the
    opinion word), forcing the model to learn order/context instead of counting
    how many times 'not' appears.

6. Encode words as integer token IDs using word2id (PAD_ID={PAD_ID}, NOT_ID={NOT_ID}),
    with vocabulary size V={V}.

7. Label rule:
    y = int(is_positive != negated)
    So:
    - positive + not negated -> positive (1)
    - positive + negated     -> negative (0)
    - negative + not negated -> negative (0)
    - negative + negated     -> positive (1)

Why this is useful:
- Sentence lengths vary.
- Word order matters (especially position of 'not').
- The data is controlled and interpretable, making model failures easy to diagnose.


In [ ]:
def make_sentence(length, opinion_pos=None):
    # One opinion word at a random position, optionally negated, plus 0-2 distractor 'not's.
    words = list(rng.choice(FILLER, size=length))

    is_positive = rng.random() < 0.5
    negated = rng.random() < 0.5
    if opinion_pos is None:
        opinion_pos = rng.integers(1, length)      # position 0 is kept free for a possible 'not'
    words[opinion_pos] = rng.choice(POSITIVE if is_positive else NEGATIVE)
    if negated:
        words[opinion_pos - 1] = "not"

    # distractor 'not's: never directly in front of the opinion word
    free = [i for i in range(length) if i not in (opinion_pos - 1, opinion_pos)]
    for i in rng.choice(free, size=rng.integers(0, 3), replace=False):
        words[i] = "not"

    label = int(is_positive != negated)             # 1 = positive meaning, 0 = negative
    return [word2id[w] for w in words], label, int(opinion_pos)


def make_text_dataset(n, min_len, max_len):
    seqs, labels, positions = [], [], []
    for _ in range(n):
        s, y, p = make_sentence(int(rng.integers(min_len, max_len + 1)))
        seqs.append(s); labels.append(y); positions.append(p)
    return seqs, np.array(labels), np.array(positions)


def decode(seq):
    return " ".join(vocab[i] for i in seq)

Sample dataset

In [ ]:
demo, demo_y, _ = make_text_dataset(6, 6, 16)
for s, y in zip(demo, demo_y):
    print(f"[{'pos' if y else 'neg'}] ({len(s):2d} words) {decode(s)}")

### 2. Speech dataset: spoken keywords as frames

Speech is usually fed to a network as a sequence of short **frames** (for example, one
frame every 10 ms), where each frame is a small feature vector such as MFCCs. We mimic
this directly:

- There are three "phonemes", `A`, `B` and `C`. Each is a characteristic 12-dimensional
  feature vector.
- There are six keywords, and each is one ordering of the three phonemes: `ABC`, `ACB`,
  `BAC`, `BCA`, `CAB` and `CBA`. This mirrors real pairs such as "tea" / "eat" or
  "yes" / "say", which use similar sounds in a different order.
- Each phoneme lasts a random number of frames (speaking rate varies).
- Each recording starts with a random amount of silence (the speaker does not start
  talking at frame 0) and ends with some silence.
- Every frame gets noise, and every utterance gets a small random speaker offset.

So the same keyword gives recordings of different lengths, and the keyword sits at a
different place inside each recording.

In [ ]:
N_FEAT = 12
PHONEMES = rng.normal(0, 1, size=(3, N_FEAT))           # prototypes for A, B, C
KEYWORDS = ["ABC", "ACB", "BAC", "BCA", "CAB", "CBA"]    # every word uses the same 3 sounds


def make_utterance(word_id, rate=1.0, lead=(0, 8), trail=(0, 5), noise=0.35):
    speaker = rng.normal(0, 0.3, size=N_FEAT)
    frames = [np.zeros((rng.integers(lead[0], lead[1] + 1), N_FEAT))]   # leading silence
    for ph in KEYWORDS[word_id]:
        dur = max(1, int(round(rng.integers(3, 7) * rate)))             # 3-6 frames at normal rate
        frames.append(np.tile(PHONEMES["ABC".index(ph)] + speaker, (dur, 1)))
    frames.append(np.zeros((rng.integers(trail[0], trail[1] + 1), N_FEAT)))
    x = np.concatenate(frames)
    return (x + rng.normal(0, noise, size=x.shape)).astype(np.float32)


def make_speech_dataset(n, **kwargs):
    labels = rng.integers(0, len(KEYWORDS), size=n)
    return [make_utterance(int(y), **kwargs) for y in labels], labels

In [ ]:
PHONEMES

In [ ]:
demo_x, demo_y = make_speech_dataset(4)
fig, axes = plt.subplots(1, 4, figsize=(13, 2.6), sharey=True)
for ax, x, y in zip(axes, demo_x, demo_y):
    ax.imshow(x.T, aspect="auto", cmap="magma", origin="lower")
    ax.set_title(f"keyword {KEYWORDS[y]} - {len(x)} frames", fontsize=9)
    ax.set_xlabel("frame (time)")
axes[0].set_ylabel("feature")
fig.suptitle("Synthetic 'spectrogram' of four utterances: different lengths, different start times")
plt.tight_layout()
plt.show()

### 3. The core problem: a DNN needs one fixed input size

A DNN's first layer is `nn.Linear(in_features, hidden)`. `in_features` is fixed when
the model is built. 
- For text, `in_features = MAX_LEN * embedding_dim`
- For speech, `in_features = MAX_FRAMES * n_features`

Let us look at how much the lengths vary.

In [ ]:
text_lens = [len(s) for s in make_text_dataset(3000, 5, 60)[0]]
speech_lens = [len(x) for x in make_speech_dataset(3000)[0]]

fig, axes = plt.subplots(1, 2, figsize=(11, 3))
axes[0].hist(text_lens, bins=28, color="tab:blue")
axes[0].set(title="Text: words per sentence", xlabel="length (tokens)", ylabel="count")
axes[1].hist(speech_lens, bins=25, color="tab:orange")
axes[1].set(title="Speech: frames per utterance", xlabel="length (frames)")
plt.tight_layout()
plt.show()

The standard fix is `pad_sequences`: 
- choose `MAX_LEN`
- pad the short sequences, 
- cut the long ones.

Keras exposes exactly these knobs (`padding="pre" | "post"`,`truncating="pre" | "post"`)

We write the same function below.

In [ ]:
def pad_or_truncate(seq, max_len, padding="post", truncating="post", pad_value=0):
    # Force one sequence (token ids or frame array) to exactly max_len steps.
    seq = np.asarray(seq)
    if len(seq) > max_len:
        seq = seq[:max_len] if truncating == "post" else seq[-max_len:]
    pad_shape = (max_len - len(seq),) + seq.shape[1:]
    pad = np.full(pad_shape, pad_value, dtype=seq.dtype)
    return np.concatenate([seq, pad]) if padding == "post" else np.concatenate([pad, seq])


s = make_sentence(9)[0]
print("original      :", decode(s))
print("post-padded 14:", decode(pad_or_truncate(s, 14)))
print("pre-padded  14:", decode(pad_or_truncate(s, 14, padding="pre")))
print("truncated    5:", decode(pad_or_truncate(s, 5)))

In [ ]:
MAX_LEN = 60
padded = np.stack([pad_or_truncate(s, MAX_LEN) for s in make_text_dataset(40, 5, 60)[0]])

plt.figure(figsize=(10, 3.2))
plt.imshow(padded == PAD_ID, aspect="auto", cmap="Greys", interpolation="nearest")
plt.title(f"40 sentences padded to MAX_LEN={MAX_LEN}: black cells are <pad>")
plt.xlabel("position"); plt.ylabel("sentence")
plt.show()
print(f"fraction of the input that is padding: {(padded == PAD_ID).mean():.0%}")

Padding to the longest sentence means a large part of every input is `<pad>`, which is
wasted computation. That is inefficient, but it is not yet the real problem. The real
problems appear when we train.

### 4. The models we compare

All models share the same word embedding size and a similar hidden size.

| model | how it gets a fixed-size input | order aware? |
| --- | --- | --- |
| `TextDNN` | pad / truncate to `MAX_LEN`, embed, **flatten**, MLP | yes, but each position has its own weights |
| `BagOfWords` | **average** the word embeddings, MLP | no |
| `TextGRU` | none: reads the sentence one word at a time | yes, same weights at every step |

`TextDNN` is the standard "DNN on text" recipe. The flatten step gives every<br>
(position, embedding dimension) pair its own weight into the hidden layer.

In [ ]:
EMB = 16

class TextDNN(nn.Module):
    def __init__(self, max_len, hidden=128):
        super().__init__()
        self.max_len = max_len
        self.emb = nn.Embedding(V, EMB, padding_idx=PAD_ID)
        self.mlp = nn.Sequential(
            nn.Flatten(),
            nn.Linear(max_len * EMB, hidden), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(hidden, 2),
        )

    def forward(self, x):                # x: (batch, max_len) - must be exactly max_len
        return self.mlp(self.emb(x))


class BagOfWords(nn.Module):
    def __init__(self, hidden=128):
        super().__init__()
        self.emb = nn.Embedding(V, EMB)
        self.mlp = nn.Sequential(nn.Linear(EMB, hidden), nn.ReLU(), nn.Linear(hidden, 2))

    def forward(self, x):                # x: (batch, T) - any T, averaged away
        return self.mlp(self.emb(x).mean(dim=1))


# GRU is the variation of RNN that is more efficient and can handle long-term dependencies better.
# It has gating mechanisms that control the flow of information,
# allowing it to retain relevant information over longer sequences.
class TextGRU(nn.Module):
    def __init__(self, hidden=64):
        super().__init__()
        self.emb = nn.Embedding(V, EMB)
        self.gru = nn.GRU(EMB, hidden, batch_first=True)
        self.out = nn.Linear(hidden, 2)

    def forward(self, x):                # x: (batch, T) - any T, read step by step
        _, h_last = self.gru(self.emb(x))
        return self.out(h_last[-1])


def count_params(model):
    return sum(p.numel() for p in model.parameters())

#### Feeding the models

- The DNN gets fixed-size arrays made with `pad_or_truncate`.
- The two sequence-friendly models get **no padding at all**. We group examples of the
  *same length* into a batch (length bucketing). Every tensor the GRU sees is real data.
  In a library you would usually use `pack_padded_sequence`, which achieves the same
  effect; bucketing makes it obvious that no `<pad>` is involved.

In [ ]:
def fixed_batches(xs, ys, max_len, batch_size=64, shuffle=True, **pad_kw):
    # Padded/truncated mini-batches for a DNN.
    X = torch.as_tensor(np.stack([pad_or_truncate(x, max_len, **pad_kw) for x in xs]))
    Y = torch.as_tensor(ys)
    order = torch.randperm(len(Y)) if shuffle else torch.arange(len(Y))
    for i in range(0, len(Y), batch_size):
        idx = order[i:i + batch_size]
        yield idx, X[idx], Y[idx]


def length_buckets(xs, ys, batch_size=64, shuffle=True):
    # Mini-batches where every sequence has the same true length: zero padding.
    by_len = defaultdict(list)
    for i, x in enumerate(xs):
        by_len[len(x)].append(i)
    batches = [idx[j:j + batch_size] for idx in by_len.values() for j in range(0, len(idx), batch_size)]
    if shuffle:
        rng.shuffle(batches)
    for idx in batches:
        X = torch.as_tensor(np.stack([np.asarray(xs[i]) for i in idx]))
        yield torch.as_tensor(idx), X, torch.as_tensor(ys[idx])

Training Code

In [ ]:
def train(model, make_batches, epochs=15, lr=3e-3):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    start = time.time()
    for epoch in range(epochs):
        model.train()
        total, n = 0.0, 0
        for _, x, y in make_batches():
            loss = F.cross_entropy(model(x), y)
            opt.zero_grad(); loss.backward(); opt.step()
            total += loss.item() * len(y); n += len(y)
    print(f"{type(model).__name__:>11}: final train loss {total / n:.3f}  ({time.time() - start:.0f}s)")
    return model


@torch.no_grad()
def predict(model, batches):
    # Per-example correctness, in the original example order.
    model.eval()
    correct = {}
    for idx, x, y in batches:
        for i, c in zip(idx.tolist(), (model(x).argmax(1) == y).tolist()):
            correct[i] = c
    return np.array([correct[i] for i in range(len(correct))])

### 5. Limitation one: truncation throws information away

Sentences are 5 to 60 words long. A common choice is to keep `MAX_LEN = 20` words
(most of the "typical" review) and cut the rest. We try both Keras options:

- `truncating="post"`: keep the **first** 20 words.
- `truncating="pre"`: keep the **last** 20 words.

The GRU reads the whole sentence, whatever its length.

In [ ]:
# Same logic as the original cell, but without lambda.

train_seqs, train_y, train_pos = make_text_dataset(8000, 5, 60)
test_seqs, test_y, test_pos = make_text_dataset(3000, 5, 60)
test_len = np.array([len(s) for s in test_seqs])

TRUNC_LEN = 20

def batches_dnn_keep_first():
    return fixed_batches(train_seqs, train_y, TRUNC_LEN, truncating="post")

def batches_dnn_keep_last():
    return fixed_batches(train_seqs, train_y, TRUNC_LEN, truncating="pre")

def batches_gru_full():
    return length_buckets(train_seqs, train_y)

dnn_keep_first = train(TextDNN(TRUNC_LEN), batches_dnn_keep_first)
dnn_keep_last = train(TextDNN(TRUNC_LEN), batches_dnn_keep_last)
gru_full = train(TextGRU(), batches_gru_full)

res_trunc = {
    "DNN, keep first 20": predict(
        dnn_keep_first,
        fixed_batches(test_seqs, test_y, TRUNC_LEN, shuffle=False, truncating="post")
    ),
    "DNN, keep last 20": predict(
        dnn_keep_last,
        fixed_batches(test_seqs, test_y, TRUNC_LEN, shuffle=False, truncating="pre")
    ),
    "GRU, full sentence": predict(
        gru_full,
        length_buckets(test_seqs, test_y, shuffle=False)
    ),
}

for name, c in res_trunc.items():
    print(f"{name:>20}: test accuracy {c.mean():.1%}")

In [ ]:
bins = [5, 20, 30, 40, 50, 61]
labels_b = [f"{lo}-{hi - 1}" for lo, hi in zip(bins[:-1], bins[1:])]
which = np.digitize(test_len, bins) - 1

fig, ax = plt.subplots(figsize=(8, 3.5))
w = 0.27
for k, (name, c) in enumerate(res_trunc.items()):
    acc = [c[which == b].mean() for b in range(len(labels_b))]
    print(f"{name:>20}: " + "  ".join(f"{l}: {a:.0%}" for l, a in zip(labels_b, acc)))
    ax.bar(np.arange(len(labels_b)) + (k - 1) * w, acc, w, label=name)
ax.axhline(0.5, color="gray", ls="--", lw=1, label="chance")
ax.set_xticks(range(len(labels_b)), labels_b)
ax.set(xlabel="sentence length (words)", ylabel="test accuracy", ylim=(0, 1.05),
       title=f"Truncating to {TRUNC_LEN} words")
ax.legend(fontsize=8, loc="lower left")
plt.tight_layout()
plt.show()

For sentences longer than 20 words, the truncated DNNs slide toward chance (50%). The reason
is simple: when the opinion word lands in the part that was cut, the model receives no
evidence at all, and it can only guess. Keeping the last words instead of the first ones
does not help; it only changes *which* sentences lose their opinion word.

Notice also that the DNNs are clearly worse than the GRU even on short sentences, where
nothing is cut. That is the second limitation, which we isolate next.

No value of `MAX_LEN` solves this. A larger `MAX_LEN` loses less text but brings in more
padding and more weights (Section 8). Real text also has no upper bound on length.

### 6. Limitation two: padding without truncation, each position must be learned separately

Suppose we avoid truncation completely and pad everything to `MAX_LEN = 60`. Now nothing is
thrown away. But a DNN's weights are tied to positions: the weights that look at position
7 are different from the weights that look at position 45. Each position must learn
"`not` + opinion word" on its own.

To expose this, we train on sentences of **5 to 30 words** (so positions 30 to 59 are
*always* `<pad>` during training), and test on sentences of **5 to 60 words**. That
situation is ordinary in practice: the sentences you see after deployment are sometimes
longer than the ones in your training set.

In [ ]:
PAD_LEN = 60
short_seqs, short_y, _ = make_text_dataset(8000, 5, 30)

dnn_pad = train(TextDNN(PAD_LEN), lambda: fixed_batches(short_seqs, short_y, PAD_LEN))
bow = train(BagOfWords(), lambda: length_buckets(short_seqs, short_y))
gru = train(TextGRU(), lambda: length_buckets(short_seqs, short_y))

res_pad = {
    "DNN, padded to 60": predict(dnn_pad, fixed_batches(test_seqs, test_y, PAD_LEN, shuffle=False)),
    "Bag of words": predict(bow, length_buckets(test_seqs, test_y, shuffle=False)),
    "GRU": predict(gru, length_buckets(test_seqs, test_y, shuffle=False)),
}
seen = test_len <= 30
print(f"{'':>18}  {'lengths 5-30 (seen)':>20}  {'lengths 31-60 (unseen)':>23}")
for name, c in res_pad.items():
    print(f"{name:>18}  {c[seen].mean():>20.1%}  {c[~seen].mean():>23.1%}")

In [ ]:
pos_bins = np.arange(0, 61, 5)
which_pos = np.digitize(test_pos, pos_bins) - 1
centers = pos_bins[:-1] + 2.5

fig, ax = plt.subplots(figsize=(8, 3.5))
for name, c in res_pad.items():
    acc = [c[which_pos == b].mean() for b in range(len(centers))]
    print(f"{name:>18}: " + " ".join(f"{a:.0%}" for a in acc))
    ax.plot(centers, acc, "o-", label=name)
ax.axvspan(30, 60, color="gray", alpha=0.12, label="positions that were only <pad> in training")
ax.axhline(0.5, color="gray", ls="--", lw=1)
ax.set(xlabel="position of the opinion word in the sentence", ylabel="test accuracy",
       ylim=(0.3, 1.02), title="Accuracy vs. where the opinion word appears")
ax.legend(fontsize=8, loc="lower left")
plt.tight_layout()
plt.show()

Three different behaviours:

- **DNN (padded)** is best at the first few positions and gets steadily worse further
  right, and around position 30 it is at chance. Nothing was cut this time: the word is in
  the input. But every position has its own weights, and each one learns only from the
  training sentences that happen to have the opinion word *there*. Early positions occur in
  every sentence, so they are well trained; later positions occur only in the longer
  sentences; positions 30 and beyond were only ever shown `<pad>`, so they learned nothing.
  The DNN reached a training loss near zero, so it memorised the training set instead of
  learning the rule.
- **Bag of words** does not care about position or length, but it cannot tell `not tasty ...
  cheap` from `tasty ... not cheap`. Averaging throws away the order, so it is stuck well
  below the other models even on short sentences.
- **GRU** keeps its accuracy at every position, including positions it never saw in
  training, because the step that reads word 45 uses exactly the same weights as the
  step that reads word 5.

#### The same sentence, only the padding moved

One more test on the padded DNN. We take short test sentences (lengths it was trained on)
and put the padding **in front** (`padding="pre"`) instead of behind. The words and their
order are unchanged; only their absolute positions shift.

In [ ]:
short_test = test_len <= 30
st_seqs = [s for s, keep in zip(test_seqs, short_test) if keep]
st_y = test_y[short_test]

post = predict(dnn_pad, fixed_batches(st_seqs, st_y, PAD_LEN, shuffle=False, padding="post")).mean()
pre = predict(dnn_pad, fixed_batches(st_seqs, st_y, PAD_LEN, shuffle=False, padding="pre")).mean()
g = predict(gru, length_buckets(st_seqs, st_y, shuffle=False)).mean()
print(f"DNN, padding after the sentence (as trained): {post:.1%}")
print(f"DNN, padding before the sentence            : {pre:.1%}")
print(f"GRU, no padding at all                      : {g:.1%}")

The DNN's prediction depends on *where in the input array* a word sits, not on what came
before it in the sentence. This is the same failure a DNN shows on images when an object is
shifted a few pixels, now along the time axis.

### 7. The same limitations on speech

We now repeat the comparison on the keyword-spotting data. For speech the workarounds are:

- DNN: pad or truncate each recording to `MAX_FRAMES = 40` frames (with zeros, which is
  silence), then flatten `40 x 12` into 480 inputs.
- Bag of frames: average all frames into one 12-dimensional vector. This is the speech
  version of bag of words.
- GRU: read the frames one at a time.

All three are trained on recordings of normal speaking rate that start within the first 8
frames. They are then tested on three conditions that happen in everyday audio:

1. **Normal**: same conditions as training.
2. **Late start**: the speaker starts after 20 to 40 frames of silence (the recording
   button was pressed early).
3. **Slow speaker**: each sound lasts twice as long.

In [ ]:
class SpeechDNN(nn.Module):
    def __init__(self, max_frames, hidden=128):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Flatten(),
            nn.Linear(max_frames * N_FEAT, hidden), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(hidden, len(KEYWORDS)),
        )

    def forward(self, x):                # x: (batch, max_frames, n_feat)
        return self.mlp(x)


class BagOfFrames(nn.Module):
    def __init__(self, hidden=128):
        super().__init__()
        self.mlp = nn.Sequential(nn.Linear(N_FEAT, hidden), nn.ReLU(), nn.Linear(hidden, len(KEYWORDS)))

    def forward(self, x):                # x: (batch, T, n_feat), averaged over T
        return self.mlp(x.mean(dim=1))


class SpeechGRU(nn.Module):
    def __init__(self, hidden=64):
        super().__init__()
        self.gru = nn.GRU(N_FEAT, hidden, batch_first=True)
        self.out = nn.Linear(hidden, len(KEYWORDS))

    def forward(self, x):                # x: (batch, T, n_feat), any T
        _, h_last = self.gru(x)
        return self.out(h_last[-1])

In [ ]:
MAX_FRAMES = 40
sp_train_x, sp_train_y = make_speech_dataset(6000)

sp_dnn = train(SpeechDNN(MAX_FRAMES), lambda: fixed_batches(sp_train_x, sp_train_y, MAX_FRAMES, pad_value=0.0))
sp_bof = train(BagOfFrames(), lambda: length_buckets(sp_train_x, sp_train_y))
sp_gru = train(SpeechGRU(), lambda: length_buckets(sp_train_x, sp_train_y))

In [ ]:
conditions = {
    "normal": make_speech_dataset(1500),
    "late start": make_speech_dataset(1500, lead=(20, 40)),
    "slow speaker": make_speech_dataset(1500, rate=2.0),
}

sp_results = defaultdict(dict)
for cond, (xs, ys) in conditions.items():
    sp_results["DNN, pad/truncate to 40"][cond] = predict(
        sp_dnn, fixed_batches(xs, ys, MAX_FRAMES, shuffle=False, pad_value=0.0)).mean()
    sp_results["Bag of frames"][cond] = predict(sp_bof, length_buckets(xs, ys, shuffle=False)).mean()
    sp_results["GRU"][cond] = predict(sp_gru, length_buckets(xs, ys, shuffle=False)).mean()
    lens = [len(x) for x in xs]
    print(f"{cond:>12}: {min(lens)}-{max(lens)} frames, "
          f"{np.mean(np.array(lens) > MAX_FRAMES):.0%} longer than MAX_FRAMES")

print()
for name, accs in sp_results.items():
    print(f"{name:>24}: " + "  ".join(f"{c}: {a:.1%}" for c, a in accs.items()))

fig, ax = plt.subplots(figsize=(8, 3.5))
w = 0.27
for k, (name, accs) in enumerate(sp_results.items()):
    vals = [accs[c] for c in conditions]
    bars = ax.bar(np.arange(len(conditions)) + (k - 1) * w, vals, w, label=name)
    ax.bar_label(bars, fmt="{:.0%}", fontsize=7)
ax.axhline(1 / len(KEYWORDS), color="gray", ls="--", lw=1, label="chance (1/6)")
ax.set_xticks(range(len(conditions)), list(conditions))
ax.set(ylabel="test accuracy", ylim=(0, 1.3), title="Keyword spotting under everyday variations")
ax.legend(fontsize=8, loc="upper center", ncol=4)
plt.tight_layout()
plt.show()

The pattern matches the text experiment:

- **Bag of frames** is near chance in every condition. All six keywords contain the same
  three sounds for (on average) the same time, so their averages are indistinguishable. The
  information is *only* in the order.
- **DNN** does well under training conditions. With a late start, the keyword moves to
  frames the DNN never learned to read, and part of it is cut off at frame 40. With a slow
  speaker, every sound is stretched and lands on different input weights, and the ending is
  often truncated.
- **GRU** reads silence, then `A`, then `B`, then `C`, whatever the timing. It was never
  shown late starts or slow speech, but the question it learned to answer ("which order did
  the sounds come in?") does not depend on absolute frame numbers.

### 8. Limitation four: DNN parameters grow with the maximum length

A longer `MAX_LEN` would reduce truncation, but the DNN's first layer grows linearly with it.
The GRU's parameter count does not depend on the sequence length at all.

In [ ]:
print(f"{'MAX_LEN':>8} {'TextDNN params':>15} {'TextGRU params':>15}")
for L in [20, 60, 200, 1000, 5000]:
    print(f"{L:>8} {count_params(TextDNN(L)):>15,} {count_params(TextGRU()):>15,}")

print(f"\n1 s of audio at 100 frames/s = 100 frames; 10 s = 1000 frames.")
print(f"SpeechDNN for 10 s clips: {count_params(SpeechDNN(1000)):,} params, "
      f"SpeechGRU for any clip length: {count_params(SpeechGRU()):,} params")

### 9. What a pure sequence model does

A recurrent network keeps a **hidden state** `h`, a fixed-size summary of everything read
so far. At every step `t` it combines the current input `x_t` with the previous state:

$$h_t = \tanh(W_x\, x_t + W_h\, h_{t-1} + b)$$

and after the last step the classifier looks at the final state. Three properties follow
directly from that single equation, and each fixes one DNN limitation:

| property of the recurrence | DNN limitation it fixes |
| --- | --- |
| The loop runs for as many steps as the input has. | No `MAX_LEN`, so no truncation and no padding. |
| `W_x`, `W_h` and `b` are reused at every step. | A rule learned at position 5 automatically applies at position 45. Parameter count is independent of length. |
| `h_t` depends on `h_{t-1}`, so on the order of the inputs. | Order is preserved, unlike averaging. |

Below is the recurrence written out by hand, with nothing hidden inside a library call. The
same three weight tensors process a 3-word sentence and a 200-word sentence.

In [ ]:
class TinyRNN(nn.Module):
    # A vanilla RNN written as an explicit loop over time.
    def __init__(self, in_dim, hidden):
        super().__init__()
        self.W_x = nn.Linear(in_dim, hidden)              # includes the bias b
        self.W_h = nn.Linear(hidden, hidden, bias=False)
        self.hidden = hidden

    def forward(self, x):                                  # x: (T, in_dim), any T
        h = torch.zeros(self.hidden)
        for x_t in x:                                      # one step per token / frame
            h = torch.tanh(self.W_x(x_t) + self.W_h(h))    # same weights every step
        return h                                           # fixed-size summary

In [ ]:
rnn = TinyRNN(in_dim=EMB, hidden=8)
emb = nn.Embedding(V, EMB)
print("RNN parameters:", count_params(rnn))

for length in [3, 17, 200]:
    sentence = torch.as_tensor(make_sentence(length)[0])
    h = rnn(emb(sentence))
    print(f"sentence of {length:>3} words -> final state shape {tuple(h.shape)}")

In practice we use a **gated** recurrent cell such as the GRU (used in all experiments above)
or the LSTM. The loop and the weight sharing are the same; the gates let the network decide
when to overwrite and when to keep its memory. That makes training over long spans much
more reliable than the plain `tanh` recurrence, which suffers from vanishing gradients.

#### Stress test: much longer than anything in training

The text GRU from Section 6 was trained on sentences of at most 30 words. A DNN cannot even
receive a 150-word sentence without truncating it. The GRU just reads it.

In [ ]:
for lo, hi in [(5, 30), (31, 60), (61, 100), (101, 150)]:
    xs, ys, _ = make_text_dataset(1000, lo, hi)
    acc = predict(gru, length_buckets(xs, ys, shuffle=False)).mean()
    print(f"sentences of {lo:>3}-{hi:<3} words: GRU accuracy {acc:.1%}")

The GRU keeps working on sentences five times longer than anything it was trained on. (On
harder tasks, a recurrent model's memory does fade over very long spans. That is the
motivation for LSTMs and, later, attention. But the length itself is never a problem for
the input layer.) That is the property we wanted: the model depends on the *content and order* of the sequence, not on its length or
on absolute positions.

The model was trained only to answer after the *complete* utterance, so its guesses in the
middle are not meaningful: it may confidently pick a word that shares the sounds heard so
far and then change its mind. Once the last sound has arrived, it settles on the correct
keyword. The point is mechanical: the model consumed the recording one frame at a time,
with no padding and no knowledge of the total length. (A real streaming recogniser is
also trained on its intermediate outputs.)

### 10. Summary

| Workaround for a DNN | What goes wrong | How a sequence model handles it |
| --- | --- | --- |
| Truncate to `MAX_LEN` | Anything past the cut is invisible. No choice of cut is safe. | Reads the whole sequence, whatever its length. |
| Pad to `MAX_LEN` | Every position has its own weights; positions that were only padding in training are never learned. Moving the padding changes the prediction. | The same weights are applied at every step, so a pattern is recognised wherever it occurs. |
| Average (bag of words / frames) | Order is lost: `not tasty ... cheap` = `tasty ... not cheap`, `ABC` = `CBA`. | The state `h_t` depends on `h_{t-1}`, so order is part of the computation. |
| Bigger `MAX_LEN` | Parameters and padding grow with `MAX_LEN`. | Parameter count is independent of length. |
| Fixed-size input | Must wait for the whole input. | Processes one step at a time, which allows streaming. |

Results from this run:

- Truncating reviews to 20 words: the DNN reached 66.9% (keep first 20) and 56.5% (keep
  last 20) accuracy, falling to 51% to 62% on sentences longer than 40 words. The GRU,
  reading whole sentences, reached 100%.
- Padding to 60 words and training on sentences of up to 30 words: the padded DNN reached
  84.2% on sentence lengths it had seen and 64.0% on longer ones, dropping to about 50%
  once the opinion word was past position 30. Moving the padding to the front of the same
  short sentences dropped it from 84.2% to 50.1%. Bag of words reached about 64% everywhere.
  The GRU reached 100% in both cases, and still 100% on 101 to 150 word sentences.
- Keyword spotting: the DNN reached 100% under training conditions, but 12.5% with a late
  start and 41.5% with a slow speaker. Bag of frames stayed at chance (about 17%) in every
  condition. The GRU reached 99.9% to 100% in all three conditions.
- Parameters: a text DNN for `MAX_LEN = 5000` needs 10.2 million weights; the GRU needs
  16,786 for any length.

These are not special properties of this dataset. They follow directly from how each model
consumes its input: a DNN binds weights to absolute positions, averaging removes order, and
a recurrent model applies one shared update rule step by step. Other sequence architectures,
such as 1D convolutions and Transformers, also avoid a fixed `MAX_LEN` through weight
sharing across positions, and they are covered separately.